## Usernames filter

This notebook aims to filter the raw usernames contained in `usernames.txt`, to keep only those created between March, 17th and September, 17th, and who let public their production.

### Chose subset of the total `usernames.txt`

In [30]:
X = 2 # Or 2 or 3

In [ ]:
with open('usernames.txt', 'r', encoding='utf-8') as f:
    raw_names = f.readlines()

breakpoint = int(len(raw_names)/3)
batch = raw_names[breakpoint*(X-1):breakpoint*(X)]
if X == 3:
    batch += raw_names[breakpoint*(X)+1:]

with open(f'covid_users_{X}.txt', 'r', encoding='utf-8') as f:
    last_done = f.readlines()[-1]
    if "Last" in last_done:
        last_done = last_done.split(": ", 1)[1]
        batch = batch[batch.index(last_done)+1 : ]
    else : 
        print('No past attempt saved.')

print(f'Remaining length batch: {len(batch)}')

Remaining length batch: 30268


### Scrap Reddit to filter each username in the batch

In [ ]:
import requests
from datetime import datetime
import time

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; scraper/1.0)"
}
session = requests.Session()
session.headers.update(headers)


start = datetime(2020, 3, 17)
end = datetime(2020, 9, 17)

to_save = []
missing = 0

for i, username in enumerate(batch):
    if i%20==0:
        print(f'Step {i}')
    time.sleep(0.5)
    try:
        r = session.get(
            f"https://www.reddit.com/user/{username.strip('\n')}/about.json", 
            timeout=3)
        r.raise_for_status()
        created_utc = r.json()["data"]["created_utc"]
    except Exception:
        missing+=1
        if missing % 5 == 0: 
            percent = (missing * 100) / (i + 1)
            print(f"Missing: {percent:.2f}%")
        continue
    date_regis = datetime.utcfromtimestamp(created_utc)
    if start <= date_regis <= end:
        r = session.get(
            f"https://www.reddit.com/user/{username.strip('\n')}/.json",
            timeout=3)
        r.raise_for_status()
        data = r.json()
        if data['data']['children']:
            to_save.append(username)
            print(f'Found:{len(to_save)}, Among:{i+1}')

print(f'Missing:{missing}')

with open(f'covid_users_{X}.txt', 'w', encoding='utf-8') as f:
    for username in to_save:
        f.write(username)

Step 0


C:\Users\cfrou\AppData\Local\Temp\ipykernel_5816\3921022883.py:34: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  date_regis = datetime.utcfromtimestamp(created_utc)


Found:1, Among:6
Found:2, Among:19
Missing: 25.00%
Step 20
Found:3, Among:26
Found:4, Among:36
Step 40
Found:5, Among:42
Missing: 21.28%
Found:6, Among:51
Step 60
Found:7, Among:63
Found:8, Among:65
Found:9, Among:68
Missing: 21.43%
Step 80
Found:10, Among:83
Missing: 21.74%
Missing: 25.77%
Step 100
Missing: 29.41%
Missing: 32.71%
Missing: 35.71%
Missing: 38.46%
Step 120
Missing: 40.98%
Missing: 43.31%
Missing: 45.45%
Missing: 47.45%
Step 140
Missing: 49.30%
Missing: 51.02%
Missing: 52.63%
Missing: 54.14%
Step 160
Missing: 55.56%
Missing: 56.89%
Missing: 58.14%
Missing: 59.32%
Step 180
Missing: 60.44%
Missing: 61.50%
Missing: 62.50%
Missing: 63.45%
Step 200
Missing: 64.36%
Missing: 65.22%
Missing: 66.04%
Missing: 66.82%
Step 220
Missing: 67.57%
Missing: 68.28%
Missing: 68.97%
Missing: 69.62%
Step 240
Missing: 70.25%
Missing: 70.85%
Missing: 71.43%
Missing: 71.98%
Step 260
Missing: 72.52%
Missing: 73.03%
Missing: 73.53%
Missing: 74.01%
Step 280
Missing: 74.47%
Missing: 74.91%
Missing: 7

In [ ]:
with open(f'covid_users_{X}.txt', 'r', encoding='utf-8') as f:
    found = f.readlines()
    if "NOT" in found[-1]:
        to_save = found[-2:] + to_save
    else : 
        found[-1] = found[-1].strip("Last (found): ")
        to_save = found + to_save

with open(f'covid_users_{X}.txt', 'w', encoding='utf-8') as f:
    for user in to_save:
        f.write(user)
    if username in to_save:
        f.write(f'Last (found): {username}')
    else : 
        f.write(f'Last (NOT found): {username}')